# RecruitView 더미 Gradio API

로컬 FastAPI와 Colab 사이의 영상 전송 및 JSON 응답을 먼저 검증합니다. 실제 모델 대신 `is_dummy: true`인 가짜 점수를 반환합니다.

In [ ]:
!pip install -q gradio

In [ ]:
from pathlib import Path

import gradio as gr

SCORE_NAMES = [
    "interview_score",
    "answer_score",
    "speaking_skills",
    "confidence_score",
    "facial_expression",
    "overall_performance",
]


def analyze_interview(video_files):
    if not video_files:
        raise gr.Error("영상 파일이 없습니다.")
    if len(video_files) != 5:
        raise gr.Error(f"영상 5개가 필요합니다. 현재 {len(video_files)}개입니다.")

    answers = []
    for index, video_path in enumerate(video_files, start=1):
        path = Path(video_path)
        # 연결 테스트임을 알아보기 쉬운 고정 더미 z-score입니다.
        scores = {
            name: round(-0.15 + index * 0.05 + score_index * 0.02, 3)
            for score_index, name in enumerate(SCORE_NAMES)
        }
        answers.append({
            "question_index": index,
            "filename": path.name,
            "size_bytes": path.stat().st_size,
            "scores": scores,
        })

    average_scores = {
        name: round(sum(answer["scores"][name] for answer in answers) / len(answers), 3)
        for name in SCORE_NAMES
    }

    return {
        "status": "success",
        "is_dummy": True,
        "video_count": len(answers),
        "answers": answers,
        "average_scores": average_scores,
    }


with gr.Blocks(title="RecruitView Dummy Analyzer") as demo:
    gr.Markdown("# RecruitView 연결 테스트")
    gr.Markdown("WebM 영상 5개를 받아 더미 점수 JSON을 반환합니다.")
    videos = gr.File(
        label="답변 영상 5개",
        file_count="multiple",
        file_types=["video"],
        type="filepath",
    )
    analyze_button = gr.Button("더미 분석")
    result = gr.JSON(label="분석 결과")
    analyze_button.click(
        fn=analyze_interview,
        inputs=videos,
        outputs=result,
        api_name="analyze",
    )


In [ ]:
demo.launch(share=True, debug=True, show_error=True)